# Lecture 2 — Class Exercise
## Bar Charts: World Happiness Report 2023

**Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.

**Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

### Rules (these will be checked in the model answer review next week)
- Every bar chart **must have a zero baseline** — no exceptions (SWD p.51)
- Every chart **must have an insight title**, not a topic title (SWD p.29)
- Aim for **professional quality** — clean background, readable font, no clutter
- Horizontal bars for long category names (SWD p.57)

## Setup — Run this cell first

In [1]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023
df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country', 'Region', 'Happiness_Score', 'GDP', 'Social_Support',
              'Life_Expectancy', 'Freedom', 'Generosity', 'Corruption']

print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

Dataset: 82 countries, 9 columns
       Country          Region  Happiness_Score    GDP  Social_Support  \
0      Finland  Western Europe             7.80  1.359           1.464   
1      Denmark  Western Europe             7.59  1.263           1.254   
2      Iceland  Western Europe             7.53  1.332           1.219   
3  Netherlands  Western Europe             7.40  1.399           1.228   
4       Sweden  Western Europe             7.34  1.214           1.347   

   Life_Expectancy  Freedom  Generosity  Corruption  
0            0.946    0.680       0.306       0.242  
1            0.880    0.649       0.302       0.141  
2            0.906    0.814       0.250       0.119  
3            0.979    0.707       0.176       0.294  
4            0.801    0.586       0.253       0.240  


In [2]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country', 'Region', 'Happiness_Score']])

Regions in dataset:
Region
Western Europe        16
Sub-Saharan Africa    15
Eastern Europe        11
Latin America         10
Middle East            8
East Asia              7
South Asia             6
Southeast Asia         5
North America          2
Oceania                2
Name: count, dtype: int64

Score range: 1.86 – 7.8

Bottom 10 countries:
         Country              Region  Happiness_Score
81   Afghanistan          South Asia             1.86
37       Lebanon         Middle East             2.39
80  Sierra Leone  Sub-Saharan Africa             3.14
79      Zimbabwe  Sub-Saharan Africa             3.20
78   Congo (DRC)  Sub-Saharan Africa             3.21
76      Botswana  Sub-Saharan Africa             3.44
77      Tanzania  Sub-Saharan Africa             3.73
64     Sri Lanka          South Asia             3.83
63    Bangladesh          South Asia             3.94
65         India          South Asia             4.04


## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.

In [3]:
# Task 1: Regional comparison bar chart
# -------------------------------------
# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # sort for horizontal bar
print(region_avg)

# Step 2: Build your chart
top_region = region_avg.iloc[-1]
bottom_region = region_avg.iloc[0]

# design choice beyond default: highlight the standout region in a different colour
bar_colors = ['#d62728' if r == top_region['Region'] else '#4C78A8'
              for r in region_avg['Region']]

fig1 = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker_color=bar_colors,
    text=region_avg['Happiness_Score'].round(2),
    textposition='outside'
))

fig1.update_layout(
    title=dict(
        text=(f"<b>{top_region['Region']} is the happiest region, scoring "
              f"{top_region['Happiness_Score'] - bottom_region['Happiness_Score']:.1f} points "
              f"above {bottom_region['Region']}</b>"),
        font=dict(size=16)
    ),
    xaxis=dict(title='Average Happiness Score',
               range=[0, region_avg['Happiness_Score'].max() * 1.15]),  # zero baseline
    yaxis=dict(title=''),
    plot_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=60, b=10),
    height=450
)
fig1.update_xaxes(showgrid=True, gridcolor='#eeeeee', zeroline=True, zerolinecolor='#cccccc')
fig1.show()

               Region  Happiness_Score
6          South Asia         3.930000
8  Sub-Saharan Africa         4.180000
3         Middle East         5.402500
7      Southeast Asia         5.710000
0           East Asia         6.018571
1      Eastern Europe         6.129091
2       Latin America         6.171000
4       North America         6.925000
9      Western Europe         7.090625
5             Oceania         7.110000


## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.

In [4]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------
# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'

bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build your chart
color_map = {'Top 8': '#2CA02C', 'Bottom 8': '#D62728'}
gap = top8['Happiness_Score'].mean() - bottom8['Happiness_Score'].mean()

fig2 = go.Figure()
for group in ['Bottom 8', 'Top 8']:
    subset = combined[combined['Group'] == group]
    fig2.add_bar(
        x=subset['Happiness_Score'],
        y=subset['Country'],
        orientation='h',
        name=group,
        marker_color=color_map[group],
        text=subset['Happiness_Score'].round(2),
        textposition='outside'
    )

# visual separator: dashed vertical line at the global average
fig2.add_vline(
    x=global_avg,
    line_dash='dash',
    line_color='gray',
    annotation_text=f'Global average ({global_avg:.2f})',
    annotation_position='top'
)

fig2.update_layout(
    title=dict(
        text=f'<b>The 8 happiest countries score {gap:.1f} points higher than the 8 least happy</b>',
        font=dict(size=16)
    ),
    xaxis=dict(title='Happiness Score',
               range=[0, combined['Happiness_Score'].max() * 1.15]),  # zero baseline
    yaxis=dict(title='', categoryorder='array', categoryarray=combined['Country']),
    barmode='overlay',
    plot_bgcolor='white',
    font=dict(family='Arial', size=12),
    legend=dict(title='Group', orientation='h', y=1.05, x=0.5, xanchor='center'),
    margin=dict(l=10, r=10, t=90, b=10),
    height=600
)
fig2.update_xaxes(showgrid=True, gridcolor='#eeeeee', zeroline=True, zerolinecolor='#cccccc')
fig2.show()

Global average: 5.74


## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

**Regions to include:** `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`

In [5]:
# Stretch goal — grouped bar chart
regions_of_interest = ['Western Europe', 'Latin America', 'East Asia',
                       'Sub-Saharan Africa', 'South Asia']

subset = df[df['Region'].isin(regions_of_interest)]
factor_avg = subset.groupby('Region')[['GDP', 'Freedom']].mean().reindex(regions_of_interest)

fig3 = go.Figure()
fig3.add_bar(name='GDP per Capita', x=factor_avg.index, y=factor_avg['GDP'], marker_color='#1F77B4')
fig3.add_bar(name='Freedom', x=factor_avg.index, y=factor_avg['Freedom'], marker_color='#FF7F0E')

fig3.update_layout(
    barmode='group',
    title=dict(
        text=('<b>Western Europe leads on both wealth and freedom — the gap with '
              'South Asia and Sub-Saharan Africa is stark</b>'),
        font=dict(size=15)
    ),
    yaxis=dict(title='Average Contribution Score', rangemode='tozero'),  # zero baseline
    xaxis=dict(title=''),
    plot_bgcolor='white',
    font=dict(family='Arial', size=12),
    legend=dict(orientation='h', y=1.15, x=0.5, xanchor='center'),
    margin=dict(l=10, r=10, t=90, b=10),
    height=450
)
fig3.show()